# Sesi 2 · Setting up RAG

Workshop Navasena × UGM

Dari PDF sampai bisa dicari:

**PDF → Markdown (Docling) → chunk → embedding → PostgreSQL + pgvector → pencarian**

Pustaka kita berisi delapan paper arXiv berlisensi CC BY tentang RAG, ditambah dua dokumen contoh buatan.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from rag import db, llm, pipeline, safety
conn = db.connect()
print("Pustaka:", db.stats(conn), "| key LLM terisi:", llm.has_key())

## 1. Dari PDF ke teks dengan Docling

PDF menyimpan posisi huruf di halaman, bukan struktur dokumen. Menyalin teks mentahnya menghasilkan kolom yang
tercampur dan tabel yang berantakan. [Docling](https://docling-project.github.io/docling/) membaca tata letak
halaman lalu menyusunnya kembali menjadi Markdown dengan judul bagian dan tabel yang utuh.

Inti kodenya hanya beberapa baris (lihat `scripts/parse_docling.py`):

```python
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
hasil = converter.convert("data/pdf/2309.15217.pdf")
markdown = hasil.document.export_to_markdown()
```

Docling membawa PyTorch dan model tata letak, jadi dipasang di image terpisah. Hasil parsing delapan paper
sudah tersedia di `data/parsed/`. Untuk mengulanginya sendiri:

```bash
docker compose --profile docling run --rm docling
```

In [ ]:
import json

papers = json.loads((ROOT / "data" / "papers.json").read_text())
for paper in papers:
    panjang = len((ROOT / "data" / "parsed" / f"{paper['id']}.md").read_text())
    print(f"{paper['id']}  {panjang:>7,} karakter  {paper['title'][:70]}")

In [ ]:
ragas = (ROOT / "data" / "parsed" / "2309.15217.md").read_text()
print(ragas[:1500])

Tabel ikut terbawa sebagai tabel Markdown, bukan deretan angka tanpa kolom:

In [ ]:
awal = ragas.index("|")
print(ragas[awal : awal + 900])

## 2. Chunking

Dokumen dipotong karena dua alasan: model embedding punya batas panjang masukan, dan potongan yang kecil
membuat pencarian lebih tepat sasaran.

Kita bandingkan dua cara:
- **naif**: potong setiap 700 karakter, tanpa melihat isi;
- **mengikuti struktur**: tidak melewati batas bagian, paragraf dijaga utuh, daftar pustaka dibuang.

In [ ]:
from rag.chunk import chunk_fixed, chunk_markdown

naif = chunk_fixed(ragas, 700)
terstruktur = chunk_markdown(ragas, max_chars=700, overlap=100)
print(f"naif: {len(naif)} chunk | mengikuti struktur: {len(terstruktur)} chunk\n")

print("=== NAIF, chunk ke-12 ===")
print(naif[11].content)
print("\n=== MENGIKUTI STRUKTUR, chunk ke-12 (bagian:", terstruktur[11].section, ") ===")
print(terstruktur[11].content)

Perhatikan awal dan akhir chunk naif: kalimatnya terpotong di tengah. Chunk seperti itu tetap bisa ditemukan,
tetapi LLM menerima kalimat yang tidak lengkap.

`overlap` menyalin ekor chunk sebelumnya ke awal chunk berikutnya:

In [ ]:
print("...", terstruktur[11].content[-120:])
print()
print(terstruktur[12].content[:120], "...")

## 3. Embedding

Model embedding mengubah teks menjadi vektor. Teks yang maknanya dekat menghasilkan vektor yang arahnya dekat.
Kita memakai `paraphrase-multilingual-MiniLM-L12-v2`: berjalan di CPU, 384 dimensi, dan multibahasa, sehingga
pertanyaan berbahasa Indonesia bisa menemukan paper berbahasa Inggris.

In [ ]:
import pandas as pd
from rag.embed import embed_texts

kalimat = [
    "Bagaimana cara mengukur kualitas jawaban RAG?",
    "How do we evaluate the quality of RAG answers?",
    "Faithfulness measures whether the answer is grounded in the retrieved context.",
    "Resep rendang membutuhkan santan dan waktu memasak yang lama.",
]
vektor = embed_texts(kalimat)
print("Bentuk matriks:", vektor.shape)

label = ["ID: ukur RAG", "EN: evaluate RAG", "EN: faithfulness", "ID: rendang"]
pd.DataFrame(vektor @ vektor.T, index=label, columns=label).round(2)

Angka di tabel adalah kemiripan kosinus (1 = searah, 0 = tidak berhubungan). Kalimat Indonesia dan terjemahan
Inggrisnya berdekatan, sedangkan kalimat tentang rendang jauh dari semuanya.

**Batas model ini 128 token.** Bagian chunk setelah itu tidak ikut terbaca oleh model embedding. Karena itu
ukuran chunk kita 700 karakter, dan judul dokumen serta judul bagian ditaruh di depan teks yang di-embed.

## 4. Menyimpan di PostgreSQL + pgvector

`pgvector` menambahkan tipe kolom `vector` dan operator jarak ke PostgreSQL. Vektor tersimpan di tabel yang sama
dengan teks dan metadatanya, jadi pencarian kemiripan bisa digabung dengan `WHERE`, `JOIN`, dan hak akses biasa.

In [ ]:
print((ROOT / "db" / "init.sql").read_text()[:1500])

Seluruh pustaka dibangun oleh `scripts/ingest.py` (sudah dijalankan saat `docker compose up`). Intinya:

```python
chunks  = chunk_markdown(markdown, max_chars=700, overlap=100)
vectors = embed_texts([f"{judul}. {c.section}. {c.content}" for c in chunks])
db.insert_chunks(conn, document_id, chunks, vectors)
```

Isi pustaka sekarang:

In [ ]:
pd.DataFrame(
    conn.execute("""
        SELECT d.source_id, d.access_level, count(*) AS chunk, left(d.title, 60) AS judul
        FROM documents d JOIN chunks c ON c.document_id = d.id
        GROUP BY d.id ORDER BY d.id
    """).fetchall(),
    columns=["sumber", "akses", "chunk", "judul"],
)

## 5. Mencari dengan SQL

Operator `<=>` menghitung jarak kosinus. Kemiripan = 1 − jarak.

In [ ]:
from rag.embed import embed_query

q = embed_query("Bagaimana Ragas mengukur faithfulness sebuah jawaban?")

hasil = conn.execute("""
    SELECT round((1 - (c.embedding <=> %(q)s))::numeric, 3) AS kemiripan,
           left(d.title, 40) AS dokumen,
           left(c.section, 30) AS bagian,
           left(c.content, 110) AS isi
    FROM chunks c JOIN documents d ON d.id = c.document_id
    WHERE d.access_level = 'public'
    ORDER BY c.embedding <=> %(q)s
    LIMIT 5
""", {"q": q}).fetchall()

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(hasil, columns=["kemiripan", "dokumen", "bagian", "isi"])

Indeks HNSW membuat pencarian tetap cepat ketika jumlah chunk mencapai jutaan. Dengan sekitar seribu baris,
PostgreSQL menilai membaca seluruh tabel lebih murah, jadi indeksnya belum dipakai. `EXPLAIN` memperlihatkan
keputusan itu, sebelum dan sesudah pembacaan seluruh tabel dimatikan:

In [ ]:
def rencana(judul):
    baris = conn.execute("EXPLAIN SELECT id FROM chunks ORDER BY embedding <=> %s LIMIT 5", (q,)).fetchall()
    print(judul)
    for (teks,) in baris:
        print("  ", teks[:95])

rencana("Bawaan:")
conn.execute("SET enable_seqscan = off")
print()
rencana("Tanpa pembacaan seluruh tabel:")
conn.execute("RESET enable_seqscan")

## 6. Melihat embedding

Vektor 384 dimensi tidak bisa digambar, jadi kita proyeksikan ke dua dimensi dengan PCA. Setiap titik adalah
satu chunk dan warnanya menunjukkan dokumen asalnya.

In [ ]:
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA

baris = conn.execute("""
    SELECT left(d.title, 45), left(c.content, 100), c.embedding::real[]
    FROM chunks c JOIN documents d ON d.id = c.document_id
""").fetchall()
peta = pd.DataFrame(baris, columns=["dokumen", "cuplikan", "embedding"])
pca = PCA(n_components=2, random_state=0)
peta[["x", "y"]] = pca.fit_transform(np.array(peta.pop("embedding").tolist()))

gambar = px.scatter(peta, x="x", y="y", color="dokumen", hover_data=["cuplikan"], opacity=0.7,
                    template="plotly_dark", height=560)
gambar.update_layout(legend={"orientation": "h", "y": -0.15, "title": None})
gambar.show()

Chunk dari dokumen yang sama cenderung mengelompok, dan dokumen dengan topik berdekatan saling bertumpang
tindih. PCA hanya menampilkan dua arah sebaran terbesar, jadi jarak di gambar adalah pendekatan kasar.

## 7. Seberapa baik pencariannya?

Untuk 18 pertanyaan contoh sudah dicatat dokumen yang seharusnya menjawabnya. Kita hitung seberapa sering
dokumen itu muncul di peringkat 1 dan di lima besar. Pengukuran ini tidak memanggil LLM.

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate_retrieval import evaluate

nilai = pd.DataFrame(evaluate(conn))
print(f"hit@1 = {(nilai['rank'] == 1).mean():.0%}   hit@5 = {nilai['rank'].notna().mean():.0%}")
nilai[nilai["rank"] != 1]

Pertanyaan yang gagal patut dibaca satu per satu. Biasanya penyebabnya salah satu dari tiga hal: istilahnya
muncul di lebih dari satu paper, jawabannya tersebar di banyak chunk, atau pertanyaannya terlalu umum.

### Coba sendiri
- Ubah `max_chars` menjadi 300 atau 1500 di sel chunking. Berapa jumlah chunk-nya, dan seperti apa isinya?
- Tambahkan kalimat Anda sendiri ke daftar `kalimat` di sel embedding.
- Ganti pertanyaan di sel pencarian SQL, lalu ubah `'public'` menjadi `'internal'`.